# RQ2 validation: automatic paired diagnostic

Creator TIB `valid` only, pinned revision `e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee`. These 20 S1/S3 summaries use a deterministic mostly extractive engine and lexical structure; this is **not** the formal controlled study. The provided abstracts are noisy proxies. ROUGE-L measures similarity to those abstracts, and source lexical overlap measures shared word types. Neither measures factuality. No blinded or other human scoring is used. The sealed `test` split is not accessed.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
root=Path.cwd()
receipt=root/'experiments/outputs/diagnostic/rq2_valid_paired/automatic_evaluation.json'
d=json.loads(receipt.read_text(encoding='utf-8'))
assert d['split']=='valid' and d['test_content_accessed'] is False and d['n']==20
print('Receipt:',receipt)
print('Status:',d['status'],'| n=',d['n'])
print('Dataset revision:',d['revision'])

In [ ]:
rows=[]
for r in d['rows']:
    rows.append({'DOI':r['doi'].split()[0], 'abstract_words':r['abstract_words'], 'reference_screen':r['reference_screen'], 'source_truncated':r['source_prefix_truncated'], 'S1_ROUGE_L':r['arms']['S1']['rouge_l_abstract_proxy'], 'S3_ROUGE_L':r['arms']['S3']['rouge_l_abstract_proxy'], 'd_ROUGE_L':r['delta_s3_minus_s1']['rouge_l_abstract_proxy'], 'S1_source_overlap':r['arms']['S1']['source_lexical_support_proxy'], 'S3_source_overlap':r['arms']['S3']['source_lexical_support_proxy'], 'd_source_overlap':r['delta_s3_minus_s1']['source_lexical_support_proxy'], 'S1_prompt_est':r['arms']['S1']['cumulative_prompt_token_estimate'], 'S3_prompt_est':r['arms']['S3']['cumulative_prompt_token_estimate']})
df=pd.DataFrame(rows)
display(df.round(4))
print('Truncated source:',int(df.source_truncated.sum()),'/20; abstract <30 words:',int((df.abstract_words<30).sum()),'/20')

In [ ]:
paired=pd.DataFrame([{'metric':k,'n':v['n'],'mean_S3_minus_S1':v['mean_s3_minus_s1'],'CI_low':v['ci95'][0],'CI_high':v['ci95'][1]} for k,v in d['paired'].items()])
display(paired.round(4))
strata=pd.DataFrame([{'subset':k,'n':v['n'],'ROUGE_delta':v['paired']['rouge_l_abstract_proxy']['mean_s3_minus_s1'],'ROUGE_CI':str([round(z,4) for z in v['paired']['rouge_l_abstract_proxy']['ci95']])} for k,v in d['strata'].items()])
display(strata)
print('Limits:', json.dumps(d['limits'],ensure_ascii=False))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].scatter(df.S1_ROUGE_L,df.S3_ROUGE_L,c=df.source_truncated.map({True:'#d97706',False:'#2563eb'}))
lo=min(df.S1_ROUGE_L.min(),df.S3_ROUGE_L.min()); hi=max(df.S1_ROUGE_L.max(),df.S3_ROUGE_L.max())
ax[0].plot([lo,hi],[lo,hi],color='gray',linestyle='--'); ax[0].set(xlabel='S1 ROUGE-L',ylabel='S3 ROUGE-L',title='Paired abstract-proxy similarity')
ax[1].hist(df.d_ROUGE_L,bins=10,color='#2563eb',alpha=.8); ax[1].axvline(0,color='black',linestyle='--'); ax[1].set(xlabel='S3 − S1 ROUGE-L',ylabel='Talks',title='Per-talk paired differences')
fig.tight_layout(); display(fig); plt.close(fig)

## Interpretation

The observed difference is descriptive and has a wide paired interval around zero. Since 13/20 inputs are prefix truncated, 13/20 pairs exceed the approximate cumulative prompt cap, 9/20 abstracts are under 30 words, and generation is deterministic/extractive, this result cannot answer whether a controlled LLM S3 system improves summary quality. The next run must use the same source span, a declared real generator, measured tokenizer usage and a frozen automatic primary outcome. The 12 `candidate_unverified` and 8 `unsuitable` abstract strata are shown above; candidate status does not verify source support. Visual S4 remains `not_run`: the cohort has keyframe timings but no aligned image pixels/OCR.